# Analysis-ready panel construction for DiD / event-study / synthetic-control prep

This notebook takes the treated quarterly municipality panel saved in `ROOT/out` and constructs the analysis-ready dataset needed for:

1. outcome engineering (`price_growth`, `contracts_per_10k`)
2. staggered treatment timing (`first_treat_period`, `treated`, `event_time`)
3. scalar controls from nested dictionaries (`Population`, `Migration`, `Income`)
4. a housing-stress proxy based on rent and income

This **fixed** version is aligned with the corrected imputation pipeline:

- it preserves the new exclusion / imputation flags
- it treats non-positive rent prices as missing for price-based analysis
- it prevents `-inf`, `inf`, and fake zero-price growth series
- it keeps municipalities with no rent series in the panel, but flags them for exclusion from **price-based** analysis

In [1]:
from pathlib import Path
import ast
import re
import unicodedata

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [2]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    return start

ROOT = find_project_root()
OUT_DIR = ROOT / "out"
OUT_DIR.mkdir(parents=True, exist_ok=True)

candidate_paths = [
    OUT_DIR / "quarterly_panel_treated_fixed.pkl",
    OUT_DIR / "quarterly_panel_treated_fixed.jsonl",
    OUT_DIR / "quarterly_panel_treated.pkl",
    OUT_DIR / "quarterly_panel_treated.jsonl",
]

print("ROOT:", ROOT)
print("OUT_DIR:", OUT_DIR)
for p in candidate_paths:
    print(f"{p.name} exists:", p.exists())

ROOT: /home/venatio/documents/university/public-sector-2026
OUT_DIR: /home/venatio/documents/university/public-sector-2026/out
quarterly_panel_treated_fixed.pkl exists: False
quarterly_panel_treated_fixed.jsonl exists: False
quarterly_panel_treated.pkl exists: True
quarterly_panel_treated.jsonl exists: True


In [3]:
panel_path = next((p for p in candidate_paths if p.exists()), None)
if panel_path is None:
    raise FileNotFoundError(
        "Could not find any treated quarterly panel in ROOT/out. "
        f"Tried: {[str(p.name) for p in candidate_paths]}"
    )

if panel_path.suffix == ".pkl":
    panel = pd.read_pickle(panel_path)
elif panel_path.suffix == ".jsonl":
    panel = pd.read_json(panel_path, lines=True)
else:
    panel = pd.read_csv(panel_path)

print("Loaded:", panel_path.name)
print("Shape:", panel.shape)
print("Columns:", panel.columns.tolist())
display(panel.head())

Loaded: quarterly_panel_treated.pkl
Shape: (56908, 20)
Columns: ['Code', 'Municipality', 'Year', 'Quarter', 'Prices', 'Contracts', 'Population', 'Migration', 'Income', 'price_observed_raw', 'contracts_observed_raw', 'price_imputed', 'contracts_imputed', 'has_any_observed_price', 'has_any_observed_contracts', 'exclude_from_price_analysis', 'exclude_from_contract_analysis', 'Treated 1', 'Treated 2', 'Control']


,Code,Municipality,Year,Quarter,Prices,Contracts,Population,Migration,Income,price_observed_raw,contracts_observed_raw,price_imputed,contracts_imputed,has_any_observed_price,has_any_observed_contracts,exclude_from_price_analysis,exclude_from_contract_analysis,Treated 1,Treated 2,Control
0,00005,NaN,2015,Q1,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}",0,0,0,0,0,0,1,1,0,0,1
1,00005,NaN,2015,Q2,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}",0,0,0,0,0,0,1,1,0,0,1
2,00005,NaN,2015,Q3,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}",0,0,0,0,0,0,1,1,0,0,1
3,00005,NaN,2015,Q4,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}",0,0,0,0,0,0,1,1,0,0,1
4,02050,NaN,2011,Q1,NaN,NaN,"{'Total': {'Total': None, 'Males': None, 'Fema...",NaN,"{'Tax Base': None, 'Tax Liability': None}",0,0,0,0,0,0,1,1,0,0,1


In [4]:
def normalize_text(value) -> str:
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()

    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))

    value = value.replace("’", "'").replace("`", "'")
    value = value.replace("-", " ")
    value = value.replace("/", " ")
    value = value.replace("'", "")

    value = re.sub(r"[^a-z0-9\s]", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


def parse_dictish(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, dict):
        return x
    x = str(x).strip()
    if x == "":
        return np.nan
    try:
        return ast.literal_eval(x)
    except Exception:
        return np.nan


def get_nested_value(d, *keys):
    cur = d
    for key in keys:
        if not isinstance(cur, dict) or key not in cur:
            return np.nan
        cur = cur[key]
    return cur


def as_numeric(series):
    return pd.to_numeric(series, errors="coerce")


def clean_pct_series(s: pd.Series) -> pd.Series:
    s = pd.to_numeric(s, errors="coerce").copy()
    s.loc[~np.isfinite(s)] = np.nan
    return s

## 1. Basic cleaning and schema alignment

This version expects the corrected treated panel and preserves the new flags:

- `price_observed_raw`
- `contracts_observed_raw`
- `price_imputed`
- `contracts_imputed`
- `has_any_observed_price`
- `has_any_observed_contracts`
- `exclude_from_price_analysis`
- `exclude_from_contract_analysis`

In [5]:
panel = panel.copy()
panel.columns = [str(c).strip() for c in panel.columns]

required_cols = ["Code", "Municipality", "Year", "Quarter", "Prices", "Contracts"]
missing_required = [c for c in required_cols if c not in panel.columns]
if missing_required:
    raise ValueError(f"Missing required columns: {missing_required}")

for col in ["Population", "Migration", "Income"]:
    if col in panel.columns:
        panel[col] = panel[col].map(parse_dictish)

panel["Code"] = panel["Code"].astype(str).str.zfill(5)
panel["Municipality"] = panel["Municipality"].astype(str).str.strip()
panel["municipality_norm"] = panel["Municipality"].map(normalize_text)

panel["Year"] = pd.to_numeric(panel["Year"], errors="coerce").astype("Int64")
panel["Prices"] = as_numeric(panel["Prices"])
panel["Contracts"] = as_numeric(panel["Contracts"])

# Safety: non-positive prices should never enter price-based analysis
panel.loc[panel["Prices"] <= 0, "Prices"] = np.nan

for col in ["Treated 1", "Treated 2", "Control"]:
    if col in panel.columns:
        panel[col] = pd.to_numeric(panel[col], errors="coerce").fillna(0).astype(int)

flag_cols = [
    "price_observed_raw",
    "contracts_observed_raw",
    "price_imputed",
    "contracts_imputed",
    "has_any_observed_price",
    "has_any_observed_contracts",
    "exclude_from_price_analysis",
    "exclude_from_contract_analysis",
]

for col in flag_cols:
    if col in panel.columns:
        panel[col] = pd.to_numeric(panel[col], errors="coerce").fillna(0).astype(int)

quarter_map = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}
panel["quarter_num"] = panel["Quarter"].map(quarter_map).astype("Int64")

bad_quarters = panel.loc[panel["quarter_num"].isna(), "Quarter"].dropna().unique().tolist()
if bad_quarters:
    raise ValueError(f"Unexpected quarter labels found: {bad_quarters}")

panel["time_index"] = panel["Year"].astype(int) * 4 + panel["quarter_num"].astype(int)
panel = panel.sort_values(["Code", "Year", "quarter_num"]).reset_index(drop=True)

print(panel[["Code", "Municipality", "Year", "Quarter", "Prices", "Contracts"]].head())

    Code Municipality  Year Quarter  Prices  Contracts
0  00005          NaN  2015      Q1     NaN        NaN
1  00005          NaN  2015      Q2     NaN        NaN
2  00005          NaN  2015      Q3     NaN        NaN
3  00005          NaN  2015      Q4     NaN        NaN
4  02050          NaN  2011      Q1     NaN        NaN


## 2. Unpack nested dictionaries into scalar controls

The regression-ready dataset should not keep controls as nested dictionaries. This section expands:

- `Population`
- `Migration`
- `Income`

into scalar numeric columns.

In [6]:
# Population scalars
if "Population" in panel.columns:
    panel["pop_total"] = panel["Population"].map(lambda d: get_nested_value(d, "Total", "Total"))
    panel["pop_males"] = panel["Population"].map(lambda d: get_nested_value(d, "Total", "Males"))
    panel["pop_females"] = panel["Population"].map(lambda d: get_nested_value(d, "Total", "Females"))

    panel["pop_native_total"] = panel["Population"].map(lambda d: get_nested_value(d, "Native", "Total"))
    panel["pop_native_males"] = panel["Population"].map(lambda d: get_nested_value(d, "Native", "Males"))
    panel["pop_native_females"] = panel["Population"].map(lambda d: get_nested_value(d, "Native", "Females"))

    panel["pop_foreign_total"] = panel["Population"].map(lambda d: get_nested_value(d, "Foreign", "Total"))
    panel["pop_foreign_males"] = panel["Population"].map(lambda d: get_nested_value(d, "Foreign", "Males"))
    panel["pop_foreign_females"] = panel["Population"].map(lambda d: get_nested_value(d, "Foreign", "Females"))

# Migration scalars
if "Migration" in panel.columns:
    panel["mig_total"] = panel["Migration"].map(lambda d: get_nested_value(d, "Total", "Total"))
    panel["mig_total_males"] = panel["Migration"].map(lambda d: get_nested_value(d, "Total", "Males"))
    panel["mig_total_females"] = panel["Migration"].map(lambda d: get_nested_value(d, "Total", "Females"))

    panel["mig_inflows"] = panel["Migration"].map(lambda d: get_nested_value(d, "Inflows", "Total"))
    panel["mig_inflows_males"] = panel["Migration"].map(lambda d: get_nested_value(d, "Inflows", "Males"))
    panel["mig_inflows_females"] = panel["Migration"].map(lambda d: get_nested_value(d, "Inflows", "Females"))

    panel["mig_outflows"] = panel["Migration"].map(lambda d: get_nested_value(d, "Outflows", "Total"))
    panel["mig_outflows_males"] = panel["Migration"].map(lambda d: get_nested_value(d, "Outflows", "Males"))
    panel["mig_outflows_females"] = panel["Migration"].map(lambda d: get_nested_value(d, "Outflows", "Females"))

# Income scalars
if "Income" in panel.columns:
    panel["income_tax_base"] = panel["Income"].map(lambda d: get_nested_value(d, "Tax Base"))
    panel["income_tax_liability"] = panel["Income"].map(lambda d: get_nested_value(d, "Tax Liability"))

scalar_cols = [
    "pop_total", "pop_males", "pop_females",
    "pop_native_total", "pop_native_males", "pop_native_females",
    "pop_foreign_total", "pop_foreign_males", "pop_foreign_females",
    "mig_total", "mig_total_males", "mig_total_females",
    "mig_inflows", "mig_inflows_males", "mig_inflows_females",
    "mig_outflows", "mig_outflows_males", "mig_outflows_females",
    "income_tax_base", "income_tax_liability",
]

existing_scalar_cols = [c for c in scalar_cols if c in panel.columns]
for c in existing_scalar_cols:
    panel[c] = as_numeric(panel[c])

display(panel[["Code", "Municipality", "Year", "Quarter"] + existing_scalar_cols[:8]].head())

,Code,Municipality,Year,Quarter,pop_total,pop_males,pop_females,pop_native_total,pop_native_males,pop_native_females,pop_foreign_total,pop_foreign_males
0,00005,NaN,2015,Q1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00005,NaN,2015,Q2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00005,NaN,2015,Q3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00005,NaN,2015,Q4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,02050,NaN,2011,Q1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Build a proper staggered-treatment structure

The bookkeeping columns `Treated 1` and `Treated 2` are converted into:

- `ever_treated`
- `cohort`
- `first_treat_period`
- `first_treat_index`
- `treated`
- `event_time`

Treatment starts:
- cohort 1: `2024-Q2`
- cohort 2: `2025-Q1`

In [7]:
for col in ["Treated 1", "Treated 2"]:
    if col not in panel.columns:
        raise ValueError(
            f"Expected column '{col}' not found. "
            "Load the treated panel or add treatment dummies first."
        )

municipality_treatment = (
    panel.groupby(["Code", "Municipality"], as_index=False)[["Treated 1", "Treated 2"]]
    .max()
)

overlap = municipality_treatment[
    (municipality_treatment["Treated 1"] == 1) & (municipality_treatment["Treated 2"] == 1)
].copy()

print("Municipalities in both treatment cohorts:", len(overlap))
if len(overlap):
    display(overlap.head())

treat1_start = 2024 * 4 + 2   # 2024-Q2
treat2_start = 2025 * 4 + 1   # 2025-Q1

municipality_treatment["cohort"] = np.select(
    [
        municipality_treatment["Treated 1"] == 1,
        municipality_treatment["Treated 2"] == 1,
    ],
    [
        "2024Q2",
        "2025Q1",
    ],
    default="Never",
)

municipality_treatment["first_treat_index"] = np.select(
    [
        municipality_treatment["Treated 1"] == 1,
        municipality_treatment["Treated 2"] == 1,
    ],
    [
        treat1_start,
        treat2_start,
    ],
    default=np.nan,
)

municipality_treatment["first_treat_period"] = np.select(
    [
        municipality_treatment["Treated 1"] == 1,
        municipality_treatment["Treated 2"] == 1,
    ],
    [
        "2024-Q2",
        "2025-Q1",
    ],
    default=pd.NA,
)

municipality_treatment["ever_treated"] = (
    (municipality_treatment["Treated 1"] == 1) | (municipality_treatment["Treated 2"] == 1)
).astype(int)

panel = panel.drop(columns=["cohort", "first_treat_index", "first_treat_period", "ever_treated"], errors="ignore")
panel = panel.merge(
    municipality_treatment[["Code", "cohort", "first_treat_index", "first_treat_period", "ever_treated"]],
    on="Code",
    how="left",
)

panel["treated"] = np.where(
    (panel["ever_treated"] == 1) & (panel["time_index"] >= panel["first_treat_index"]),
    1,
    0,
).astype(int)

panel["event_time"] = np.where(
    panel["ever_treated"] == 1,
    panel["time_index"] - panel["first_treat_index"],
    np.nan,
)

display(panel[[
    "Code", "Municipality", "Year", "Quarter", "Treated 1", "Treated 2",
    "cohort", "first_treat_period", "ever_treated", "treated", "event_time"
]].head(12))

Municipalities in both treatment cohorts: 0


,Code,Municipality,Year,Quarter,Treated 1,Treated 2,cohort,first_treat_period,ever_treated,treated,event_time
0,00005,NaN,2015,Q1,0,0,NaN,NaN,NaN,0,NaN
1,00005,NaN,2015,Q2,0,0,NaN,NaN,NaN,0,NaN
2,00005,NaN,2015,Q3,0,0,NaN,NaN,NaN,0,NaN
3,00005,NaN,2015,Q4,0,0,NaN,NaN,NaN,0,NaN
4,02050,NaN,2011,Q1,0,0,NaN,NaN,NaN,0,NaN
5,02050,NaN,2011,Q2,0,0,NaN,NaN,NaN,0,NaN
6,02050,NaN,2011,Q3,0,0,NaN,NaN,NaN,0,NaN
7,02050,NaN,2011,Q4,0,0,NaN,NaN,NaN,0,NaN
8,02050,NaN,2012,Q1,0,0,NaN,NaN,NaN,0,NaN
9,02050,NaN,2012,Q2,0,0,NaN,NaN,NaN,0,NaN


## 4. Engineer the proposal's core outcomes

This fixed version computes price-based variables only when rents are valid for analysis.

- municipalities with `exclude_from_price_analysis == 1` are kept in the dataset, but
  their price-based outcomes are left missing
- non-positive rent prices are not logged
- non-finite growth values are reset to missing

In [8]:
panel = panel.sort_values(["Code", "Year", "quarter_num"]).reset_index(drop=True)

if "exclude_from_price_analysis" not in panel.columns:
    panel["exclude_from_price_analysis"] = 0

if "exclude_from_contract_analysis" not in panel.columns:
    panel["exclude_from_contract_analysis"] = 0

panel["has_valid_price_for_analysis"] = (
    panel["Prices"].notna() & (panel["exclude_from_price_analysis"] == 0)
).astype(int)

panel["has_valid_contracts_for_analysis"] = (
    panel["Contracts"].notna() & (panel["exclude_from_contract_analysis"] == 0)
).astype(int)

# Safe logs
panel["log_prices"] = np.where(
    (panel["Prices"] > 0) & (panel["exclude_from_price_analysis"] == 0),
    np.log(panel["Prices"]),
    np.nan,
)

# Price growth
panel["price_growth_qoq_pct"] = (
    panel.groupby("Code")["Prices"]
    .transform(lambda s: s.pct_change(1, fill_method=None) * 100)
)

panel["price_growth_yoy_pct"] = (
    panel.groupby("Code")["Prices"]
    .transform(lambda s: s.pct_change(4, fill_method=None) * 100)
)

panel["log_price_growth_qoq"] = (
    panel.groupby("Code")["log_prices"]
    .transform(lambda s: s.diff(1) * 100)
)

panel["log_price_growth_yoy"] = (
    panel.groupby("Code")["log_prices"]
    .transform(lambda s: s.diff(4) * 100)
)

for col in [
    "price_growth_qoq_pct",
    "price_growth_yoy_pct",
    "log_price_growth_qoq",
    "log_price_growth_yoy",
]:
    panel[col] = clean_pct_series(panel[col])

# Blank out all price-based outputs for municipalities excluded from price analysis
price_based_cols = [
    "log_prices",
    "price_growth_qoq_pct",
    "price_growth_yoy_pct",
    "log_price_growth_qoq",
    "log_price_growth_yoy",
]
panel.loc[panel["exclude_from_price_analysis"] == 1, price_based_cols] = np.nan

# Market activity
panel["contracts_per_10k"] = np.where(
    (panel["pop_total"] > 0) & panel["Contracts"].notna(),
    10000 * panel["Contracts"] / panel["pop_total"],
    np.nan,
)

panel["contracts_per_10k_qoq_pct"] = (
    panel.groupby("Code")["contracts_per_10k"]
    .transform(lambda s: s.pct_change(1, fill_method=None) * 100)
)

panel["contracts_per_10k_yoy_pct"] = (
    panel.groupby("Code")["contracts_per_10k"]
    .transform(lambda s: s.pct_change(4, fill_method=None) * 100)
)

for col in [
    "contracts_per_10k_qoq_pct",
    "contracts_per_10k_yoy_pct",
]:
    panel[col] = clean_pct_series(panel[col])

# Optional: blank out contract-based outputs only for municipalities with no contract series
contract_based_cols = [
    "contracts_per_10k",
    "contracts_per_10k_qoq_pct",
    "contracts_per_10k_yoy_pct",
]
panel.loc[panel["exclude_from_contract_analysis"] == 1, contract_based_cols] = np.nan

# Convenient aliases for baseline estimation
panel["outcome_price_growth"] = panel["price_growth_yoy_pct"]
panel["outcome_market_activity"] = panel["contracts_per_10k"]

display(panel[[
    "Code", "Municipality", "Year", "Quarter", "Prices",
    "exclude_from_price_analysis", "log_prices",
    "price_growth_qoq_pct", "price_growth_yoy_pct",
    "Contracts", "contracts_per_10k"
]].head(12))

,Code,Municipality,Year,Quarter,Prices,exclude_from_price_analysis,log_prices,price_growth_qoq_pct,price_growth_yoy_pct,Contracts,contracts_per_10k
0,00005,NaN,2015,Q1,NaN,1,NaN,NaN,NaN,NaN,NaN
1,00005,NaN,2015,Q2,NaN,1,NaN,NaN,NaN,NaN,NaN
2,00005,NaN,2015,Q3,NaN,1,NaN,NaN,NaN,NaN,NaN
3,00005,NaN,2015,Q4,NaN,1,NaN,NaN,NaN,NaN,NaN
4,02050,NaN,2011,Q1,NaN,1,NaN,NaN,NaN,NaN,NaN
5,02050,NaN,2011,Q2,NaN,1,NaN,NaN,NaN,NaN,NaN
6,02050,NaN,2011,Q3,NaN,1,NaN,NaN,NaN,NaN,NaN
7,02050,NaN,2011,Q4,NaN,1,NaN,NaN,NaN,NaN,NaN
8,02050,NaN,2012,Q1,NaN,1,NaN,NaN,NaN,NaN,NaN
9,02050,NaN,2012,Q2,NaN,1,NaN,NaN,NaN,NaN,NaN


## 5. Create a housing-stress control from rent and income

These are only computed where price-based analysis is valid and income is positive.

In [9]:
valid_income_base = panel["income_tax_base"] > 0
valid_income_liability = panel["income_tax_liability"] > 0
valid_price = panel["Prices"].notna() & (panel["exclude_from_price_analysis"] == 0)

panel["rent_to_income_proxy"] = np.where(
    valid_price & valid_income_base,
    panel["Prices"] / panel["income_tax_base"],
    np.nan,
)

panel["annualized_rent_to_income_proxy"] = np.where(
    valid_price & valid_income_base,
    (12 * panel["Prices"]) / panel["income_tax_base"],
    np.nan,
)

panel["rent_to_tax_liability_proxy"] = np.where(
    valid_price & valid_income_liability,
    panel["Prices"] / panel["income_tax_liability"],
    np.nan,
)

display(panel[[
    "Code", "Municipality", "Year", "Quarter", "Prices",
    "income_tax_base", "income_tax_liability",
    "rent_to_income_proxy", "annualized_rent_to_income_proxy"
]].head(12))

,Code,Municipality,Year,Quarter,Prices,income_tax_base,income_tax_liability,rent_to_income_proxy,annualized_rent_to_income_proxy
0,00005,NaN,2015,Q1,NaN,NaN,NaN,NaN,NaN
1,00005,NaN,2015,Q2,NaN,NaN,NaN,NaN,NaN
2,00005,NaN,2015,Q3,NaN,NaN,NaN,NaN,NaN
3,00005,NaN,2015,Q4,NaN,NaN,NaN,NaN,NaN
4,02050,NaN,2011,Q1,NaN,NaN,NaN,NaN,NaN
5,02050,NaN,2011,Q2,NaN,NaN,NaN,NaN,NaN
6,02050,NaN,2011,Q3,NaN,NaN,NaN,NaN,NaN
7,02050,NaN,2011,Q4,NaN,NaN,NaN,NaN,NaN
8,02050,NaN,2012,Q1,NaN,NaN,NaN,NaN,NaN
9,02050,NaN,2012,Q2,NaN,NaN,NaN,NaN,NaN


## 6. Assemble the analysis-ready panel

This version keeps identifiers, treatment timing, raw outcomes, engineered outcomes,
scalar controls, and the new exclusion / imputation flags.

The original nested-dict columns are dropped from the saved analysis file to avoid issues
in downstream estimation.

In [10]:
analysis_cols = [
    "Code", "Municipality", "municipality_norm", "Year", "Quarter", "quarter_num", "time_index",
    "Prices", "Contracts",

    "price_observed_raw", "contracts_observed_raw",
    "price_imputed", "contracts_imputed",
    "has_any_observed_price", "has_any_observed_contracts",
    "exclude_from_price_analysis", "exclude_from_contract_analysis",
    "has_valid_price_for_analysis", "has_valid_contracts_for_analysis",

    "Treated 1", "Treated 2", "Control",
    "ever_treated", "cohort", "first_treat_period", "first_treat_index", "treated", "event_time",

    "pop_total", "pop_males", "pop_females",
    "pop_native_total", "pop_native_males", "pop_native_females",
    "pop_foreign_total", "pop_foreign_males", "pop_foreign_females",

    "mig_total", "mig_total_males", "mig_total_females",
    "mig_inflows", "mig_inflows_males", "mig_inflows_females",
    "mig_outflows", "mig_outflows_males", "mig_outflows_females",

    "income_tax_base", "income_tax_liability",

    "log_prices",
    "price_growth_qoq_pct", "price_growth_yoy_pct",
    "log_price_growth_qoq", "log_price_growth_yoy",

    "contracts_per_10k", "contracts_per_10k_qoq_pct", "contracts_per_10k_yoy_pct",

    "rent_to_income_proxy", "annualized_rent_to_income_proxy", "rent_to_tax_liability_proxy",

    "outcome_price_growth", "outcome_market_activity",
]

analysis_cols = [c for c in analysis_cols if c in panel.columns]

analysis_ready = (
    panel[analysis_cols]
    .sort_values(["Code", "Year", "quarter_num"])
    .reset_index(drop=True)
)

dup_count = analysis_ready.duplicated(["Code", "Year", "Quarter"]).sum()
print("Duplicate Code-Year-Quarter rows:", int(dup_count))

if dup_count:
    display(
        analysis_ready[analysis_ready.duplicated(["Code", "Year", "Quarter"], keep=False)]
        .sort_values(["Code", "Year", "Quarter"])
        .head(20)
    )

display(analysis_ready.head(12))

Duplicate Code-Year-Quarter rows: 0


,Code,Municipality,municipality_norm,Year,Quarter,quarter_num,time_index,Prices,Contracts,price_observed_raw,contracts_observed_raw,price_imputed,contracts_imputed,has_any_observed_price,has_any_observed_contracts,exclude_from_price_analysis,exclude_from_contract_analysis,has_valid_price_for_analysis,has_valid_contracts_for_analysis,Treated 1,Treated 2,Control,ever_treated,cohort,first_treat_period,first_treat_index,treated,event_time,pop_total,pop_males,pop_females,pop_native_total,pop_native_males,pop_native_females,pop_foreign_total,pop_foreign_males,pop_foreign_females,mig_total,mig_total_males,mig_total_females,mig_inflows,mig_inflows_males,mig_inflows_females,mig_outflows,mig_outflows_males,mig_outflows_females,income_tax_base,income_tax_liability,log_prices,price_growth_qoq_pct,price_growth_yoy_pct,log_price_growth_qoq,log_price_growth_yoy,contracts_per_10k,contracts_per_10k_qoq_pct,contracts_per_10k_yoy_pct,rent_to_income_proxy,annualized_rent_to_income_proxy,rent_to_tax_liability_proxy,outcome_price_growth,outcome_market_activity
0,00005,NaN,,2015,Q1,1,8061,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,00005,NaN,,2015,Q2,2,8062,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,00005,NaN,,2015,Q3,3,8063,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,00005,NaN,,2015,Q4,4,8064,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,02050,NaN,,2011,Q1,1,8045,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,02050,NaN,,2011,Q2,2,8046,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,02050,NaN,,2011,Q3,3,8047,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,02050,NaN,,2011,Q4,4,8048,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,02050,NaN,,2012,Q1,1,8049,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,02050,NaN,,2012,Q2,2,8050,NaN,NaN,0,0,0,0,0,0,1,1,0,0,0,0,1,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
print("Rows:", len(analysis_ready))
print("Municipalities:", analysis_ready["Code"].nunique())
print("Years:", int(analysis_ready["Year"].min()), "to", int(analysis_ready["Year"].max()))
print()

print("Cohort counts:")
print(
    analysis_ready.groupby("Code")[["cohort"]].first()["cohort"]
    .value_counts(dropna=False)
    .sort_index()
)

print()
print("Municipalities excluded from price analysis:")
print(
    analysis_ready.groupby("Code")[["exclude_from_price_analysis"]].first()["exclude_from_price_analysis"]
    .value_counts(dropna=False)
    .sort_index()
)

print()
print("Municipalities excluded from contract analysis:")
print(
    analysis_ready.groupby("Code")[["exclude_from_contract_analysis"]].first()["exclude_from_contract_analysis"]
    .value_counts(dropna=False)
    .sort_index()
)

print()
print("Share treated by year-quarter:")
display(
    analysis_ready.groupby(["Year", "Quarter"])["treated"].mean().reset_index().tail(12)
)

print()
print("Missing outcome_price_growth:", int(analysis_ready["outcome_price_growth"].isna().sum()))
print("Missing outcome_market_activity:", int(analysis_ready["outcome_market_activity"].isna().sum()))

Rows: 56908
Municipalities: 950
Years: 2011 to 2025

Cohort counts:
cohort
2024Q2    140
2025Q1    131
Never     677
NaN         2
Name: count, dtype: int64

Municipalities excluded from price analysis:
exclude_from_price_analysis
0    528
1    422
Name: count, dtype: int64

Municipalities excluded from contract analysis:
exclude_from_contract_analysis
0    948
1      2
Name: count, dtype: int64

Share treated by year-quarter:


,Year,Quarter,treated
48,2023,Q1,0.000000
49,2023,Q2,0.000000
50,2023,Q3,0.000000
51,2023,Q4,0.000000
52,2024,Q1,0.000000
53,2024,Q2,0.147679
54,2024,Q3,0.147679
55,2024,Q4,0.147679
56,2025,Q1,0.285865
57,2025,Q2,0.285865



Missing outcome_price_growth: 27340
Missing outcome_market_activity: 88


In [12]:
OUT_PKL = OUT_DIR / "analysis_ready_panel.pkl"
OUT_JSONL = OUT_DIR / "analysis_ready_panel.jsonl"
OUT_CSV = OUT_DIR / "analysis_ready_panel.csv"

analysis_ready.to_pickle(OUT_PKL)
analysis_ready.to_json(OUT_JSONL, orient="records", lines=True, force_ascii=False)
analysis_ready.to_csv(OUT_CSV, index=False)

print("Saved:")
print(" -", OUT_PKL)
print(" -", OUT_JSONL)
print(" -", OUT_CSV)

Saved:
 - /home/venatio/documents/university/public-sector-2026/out/analysis_ready_panel.pkl
 - /home/venatio/documents/university/public-sector-2026/out/analysis_ready_panel.jsonl
 - /home/venatio/documents/university/public-sector-2026/out/analysis_ready_panel.csv
